In [6]:
import torch
import torchvision.datasets
import torch.nn.functional as F


import transformers
print(f"Cuda available: {torch.cuda.is_available()}")

# Load the STL10 dataset (train and test splits)
data_train = torchvision.datasets.STL10(root='./data', split='train', download=True)
data_test = torchvision.datasets.STL10(root='./data', split='test', download=True)

C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cuda available: True


Check train/test split

In [5]:
print(f"Train / Test Split Size  : {len(data_train):,} / {len(data_test):,}")

Train / Test Split Size  : 5,000 / 8,000


Label Encoding

In [2]:
num_classes = len(data_train.classes)
print(f"Number of classes: {num_classes}")

Number of classes: 10


In [3]:
label_tensor = torch.tensor(data_train.labels, dtype=torch.long)
print(f"Label tensor shape: {label_tensor.shape}")

Label tensor shape: torch.Size([5000])


In [4]:
onehot_labels = F.one_hot(label_tensor, num_classes=num_classes)
print(f"One-hot labels shape: {onehot_labels.shape}")

One-hot labels shape: torch.Size([5000, 10])


### Extraction of photo features with DINOv2

DINOv2 models produce high-performance visual features that can be directly employed with classifiers as simple as linear layers on a variety of computer vision tasks; these visual features are robust and perform well across domains without any requirement for fine-tuning. The models were pretrained on a dataset of 142 M images without using any labels or annotations.

https://github.com/facebookresearch/dinov2

https://dinov2.metademolab.com/

In [8]:
model_name = "facebook/dinov2-base"
model = transformers.AutoModel.from_pretrained(model_name)

Loading weights: 100%|██████████| 223/223 [00:00<00:00, 4488.21it/s]


In [9]:
#here we load processor for DinoV2
processor = transformers.AutoImageProcessor.from_pretrained(model_name)

In [10]:
#and again check GPU
model = model.to('cuda' if torch.cuda.is_available() else 'cpu')

model.eval()

Dinov2Model(
  (embeddings): Dinov2Embeddings(
    (patch_embeddings): Dinov2PatchEmbeddings(
      (projection): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): Dinov2Encoder(
    (layer): ModuleList(
      (0-11): 12 x Dinov2Layer(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attention): Dinov2Attention(
          (attention): Dinov2SelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
          )
          (output): Dinov2SelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (layer_scale1): Dinov2LayerScale()
        (drop_path): Identity()
        (norm2): LayerNorm((768,), eps=1e-06,

In [12]:
from tqdm import tqdm